# Time Series Analysis — Concepts & Synthetic Data
### Topics Covered:
- Introduction to Time Series
- Components: Trend, Seasonal, Cyclical, Residual
- Stationarity & Differencing (ADF Test)
- ACF & PACF Plots
- AR, MA, ARIMA, SARIMA Models
- Evaluation Metrics: MAE, RMSE, MAPE

### Keywords Before You Start
- **Time Series**: Data points collected at regular time intervals (daily, monthly, yearly)
- **statsmodels**: Python library for statistical models including ARIMA, SARIMA
- **datetime**: Python module to work with dates and times
- **warnings**: Module to suppress unnecessary output warnings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# statsmodels — time series modelling library
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error

print("All libraries imported successfully!")
print(f"pandas version: {pd.__version__}")

---
## 1. Introduction to Time Series

### Time Series kya hai?
A **time series** is a sequence of data points recorded at **equal time intervals**.

**Key property:** Order matters! Unlike regular tabular data, we cannot shuffle rows.

### Examples:
| Domain | Time Series Example |
|--------|-------------------|
| Finance | Daily stock prices, monthly GDP |
| Weather | Daily temperature, hourly rainfall |
| Retail | Weekly sales, monthly revenue |
| Healthcare | Daily patient count, yearly birth rate |
| Transport | Hourly traffic count, daily flight bookings |

### Why is Time Series different from normal ML?
| Normal ML (Random Forest, etc.) | Time Series |
|---------------------------------|------------|
| Rows are independent | Each row depends on previous rows |
| Shuffle data for train/test | NEVER shuffle — order is everything |
| Any split is OK | Train = old data, Test = recent data |
| Features (X) predict target (y) | Past values predict future values |

### What we want from Time Series:
1. **Understand** past behavior (components)
2. **Forecast** future values (AR, MA, ARIMA, SARIMA)

---
## 2. Creating Synthetic Time Series Data

### Keywords:
- **np.arange**: Creates evenly spaced numbers (used as time index)
- **pd.date_range**: Creates a sequence of dates with a fixed frequency
- **freq='MS'**: Month Start — first day of each month
- **Trend component**: Long-term upward or downward movement
- **Seasonal component**: Repeating pattern at fixed intervals (e.g., every 12 months)
- **Noise component**: Random fluctuations (np.random.normal)

In [ ]:
np.random.seed(42)
n = 120  # 10 years of monthly data

# Create date index
dates = pd.date_range(start='2014-01-01', periods=n, freq='MS')

# Components
t = np.arange(n)
trend     = 50 + 0.8 * t                       # upward trend
seasonal  = 20 * np.sin(2 * np.pi * t / 12)    # yearly cycle
cyclical  = 10 * np.sin(2 * np.pi * t / 48)    # 4-year business cycle
noise     = np.random.normal(0, 5, n)           # random noise

# Combine into final series
sales = trend + seasonal + cyclical + noise

# Create DataFrame
df = pd.DataFrame({'date': dates, 'sales': sales})
df.set_index('date', inplace=True)

print("Dataset shape:", df.shape)
print("Date range:", df.index[0], "to", df.index[-1])
print("\nFirst 5 rows:")
print(df.head())
print("\nBasic statistics:")
print(df.describe().round(2))

---
### Visualizing the Time Series

**Rolling Mean / Moving Average**: Average of a sliding window of data points.
Helps smooth out noise to reveal the underlying trend.
- `window=12` → 12-month rolling mean (smooths seasonal fluctuations)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: raw series + rolling mean
axes[0].plot(df.index, df['sales'], alpha=0.7, label='Monthly Sales', color='steelblue')
axes[0].plot(df.index, df['sales'].rolling(12).mean(), color='red',
             linewidth=2, label='12-Month Rolling Mean')
axes[0].set_title('Monthly Sales — Raw Data with Rolling Mean')
axes[0].set_ylabel('Sales')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bottom: rolling std (volatility)
axes[1].plot(df.index, df['sales'].rolling(12).std(), color='orange', label='Rolling Std Dev')
axes[1].set_title('Rolling Standard Deviation (12-month window)')
axes[1].set_ylabel('Std Dev')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Components of Time Series

Every time series can be broken into **4 components**:

| Component | Symbol | Description | Example |
|-----------|--------|-------------|---------|
| **Trend (T)** | T | Long-term direction (up/down/flat) | Sales growing year on year |
| **Seasonal (S)** | S | Fixed, repeating pattern | Ice cream sales peak every summer |
| **Cyclical (C)** | C | Irregular long-term waves (business cycles) | Economy expanding/contracting |
| **Residual (R)** | R | Random noise after removing T, S, C | Unexpected events |

### Two Models of Decomposition:
- **Additive**: `Y = T + S + C + R` → Use when seasonal variation is *constant*
- **Multiplicative**: `Y = T × S × C × R` → Use when seasonal variation *grows with trend*

### Keywords:
- **seasonal_decompose()**: statsmodels function to split a series into components
- **model='additive'**: For constant seasonal swings
- **model='multiplicative'**: For growing seasonal swings
- **period=12**: Tells the function our data has yearly seasonality (12 months)

In [ ]:
# Decompose the synthetic sales data
result = seasonal_decompose(df['sales'], model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))

result.observed.plot(ax=axes[0], color='steelblue', title='Observed (Original)')
axes[0].set_ylabel('Observed')

result.trend.plot(ax=axes[1], color='green', title='Trend Component')
axes[1].set_ylabel('Trend')

result.seasonal.plot(ax=axes[2], color='orange', title='Seasonal Component')
axes[2].set_ylabel('Seasonal')

result.resid.plot(ax=axes[3], color='red', title='Residual (Noise)')
axes[3].set_ylabel('Residual')
axes[3].axhline(0, linestyle='--', color='black', alpha=0.5)

plt.tight_layout()
plt.show()

# Check how much variance each component explains
print(f"Trend range: {result.trend.dropna().max() - result.trend.dropna().min():.1f}")
print(f"Seasonal range: {result.seasonal.max() - result.seasonal.min():.1f}")
print(f"Residual std: {result.resid.dropna().std():.2f}")

---
## 4. Stationarity — The Most Important Concept

### Stationary kya hota hai?
A time series is **stationary** if its statistical properties (mean, variance) do not change over time.

**AR, MA, ARIMA need stationary data to work!**

| Property | Stationary | Non-Stationary |
|----------|-----------|----------------|
| Mean | Constant | Changes (trending) |
| Variance | Constant | Changes (heteroscedastic) |
| ACF | Drops quickly | Drops very slowly |

### ADF Test (Augmented Dickey-Fuller):
**Null Hypothesis (H₀):** Series has a unit root → non-stationary
**Alternative (H₁):** Series is stationary

| p-value | Decision |
|---------|---------|
| p < 0.05 | Reject H₀ → **Stationary** ✓ |
| p ≥ 0.05 | Fail to reject H₀ → **Non-Stationary** ✗ |

### Keywords:
- **Unit root**: Mathematical property indicating non-stationarity
- **Differencing**: Subtract previous value → `y'(t) = y(t) - y(t-1)` → removes trend
- **d parameter (ARIMA)**: How many times to difference to get stationarity

In [ ]:
def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    print(f"=== ADF Test: {name} ===")
    print(f"ADF Statistic : {result[0]:.4f}")
    print(f"p-value       : {result[1]:.4f}")
    if result[1] < 0.05:
        print("Result → STATIONARY (p < 0.05) ✓\n")
    else:
        print("Result → NON-STATIONARY (p >= 0.05) — differencing needed\n")

# Test original series
adf_test(df['sales'], 'Original Sales')

# Apply 1st order differencing to remove trend
df['sales_diff1'] = df['sales'].diff()
adf_test(df['sales_diff1'].dropna(), '1st Order Difference')

# Visualize before and after
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
df['sales'].plot(ax=axes[0], title='Original Sales (Non-Stationary)', color='steelblue')
axes[0].grid(True, alpha=0.3)
df['sales_diff1'].dropna().plot(ax=axes[1], title='After 1st Differencing (Stationary)', color='green')
axes[1].axhline(0, linestyle='--', color='red', alpha=0.5)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. ACF and PACF — Choosing AR and MA Orders

### ACF — AutoCorrelation Function:
Measures how correlated the series is with its **own past values** at different lags.
→ Helps determine **q (MA order)**: look at where ACF cuts off after a spike.

### PACF — Partial AutoCorrelation Function:
Measures correlation with lag k, **after removing effects of shorter lags**.
→ Helps determine **p (AR order)**: look at where PACF cuts off after a spike.

### How to read ACF/PACF plots:
| Pattern | Model Suggested |
|---------|----------------|
| PACF cuts off at lag p, ACF decays gradually | AR(p) |
| ACF cuts off at lag q, PACF decays gradually | MA(q) |
| Both decay gradually | ARIMA(p,d,q) |
| Spikes at seasonal lags (12, 24...) | SARIMA |

### Keywords:
- **Lag k**: k time steps back (lag 1 = yesterday, lag 12 = same month last year)
- **Confidence band (blue shaded area)**: Correlations inside are NOT significant
- **Cuts off**: Spikes suddenly drop inside the band after lag k

In [ ]:
# Use stationary (differenced) series for ACF/PACF
series_stationary = df['sales_diff1'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(series_stationary, lags=30, alpha=0.05, ax=axes[0],
         title='ACF — Moving Average order (q)')
plot_pacf(series_stationary, lags=30, alpha=0.05, ax=axes[1],
          title='PACF — Autoregressive order (p)', method='ywm')

plt.tight_layout()
plt.show()

# Numerical ACF values for first 15 lags
acf_vals = acf(series_stationary, nlags=15)
pacf_vals = pacf(series_stationary, nlags=15, method='ywm')
print("ACF at lags 1-5:", [round(x,3) for x in acf_vals[1:6]])
print("PACF at lags 1-5:", [round(x,3) for x in pacf_vals[1:6]])
print("\nInterpretation:")
print(" → PACF drops sharply after lag 2 → AR(2) suggested")
print(" → ACF decays gradually → confirms AR component")

---
## 6. Train-Test Split for Time Series

### CRITICAL RULE: Never shuffle time series data!
- **Train set** = older data (used to fit the model)
- **Test set** = recent data (used to evaluate forecasts)
- Always split by **time**, not randomly

### Evaluation Metrics:
| Metric | Formula | Interpretation |
|--------|---------|---------------|
| **MAE** | mean(|actual - predicted|) | Average absolute error (same units) |
| **RMSE** | sqrt(mean((actual-predicted)²)) | Penalizes large errors more |
| **MAPE** | mean(|actual-pred|/|actual|) × 100 | % error — scale-independent |

In [ ]:
# Split: 80% train, 20% test
split = int(len(df) * 0.8)
train = df['sales'][:split]
test  = df['sales'][split:]

print(f"Train size: {len(train)} months ({train.index[0].date()} → {train.index[-1].date()})")
print(f"Test size:  {len(test)} months ({test.index[0].date()} → {test.index[-1].date()})")

def evaluate(actual, predicted, model_name):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f"\n{model_name}:")
    print(f"  MAE  = {mae:.2f}")
    print(f"  RMSE = {rmse:.2f}")
    print(f"  MAPE = {mape:.2f}%")
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results_summary = {}  # collect all model results

---
## 7. AR Model — AutoRegressive Model

### AR(p) kya hota hai?
**AR(p)** predicts the next value using **p previous values** of the same series.

**Formula:** `y(t) = c + φ₁·y(t-1) + φ₂·y(t-2) + ... + φₚ·y(t-p) + ε(t)`

Where:
- `y(t)` = value at time t (what we want to predict)
- `φ₁, φ₂, ...` = AR coefficients (learned by the model)
- `c` = constant (intercept)
- `ε(t)` = white noise error
- `p` = number of past lags used (the **order**)

### Real-life analogy:
"Today's temperature = weighted average of yesterday's and day-before-yesterday's temperature + some random error"

### Keywords:
- **AutoReg**: statsmodels class for AR model
- **lags=p**: How many past values to use (p order)
- **trend='c'**: Include a constant term
- **model.predict(start, end)**: Forecast for given time range

In [ ]:
# AR Model — use p=2 (from PACF analysis)
p = 2
ar_model = AutoReg(train, lags=p, trend='c')
ar_fit = ar_model.fit()

print(ar_fit.summary().tables[0])
print("\nAR Coefficients:", dict(zip(ar_fit.params.index, ar_fit.params.round(4).values)))

# Forecast on test set
ar_pred = ar_fit.predict(start=len(train), end=len(train)+len(test)-1)
ar_pred.index = test.index

results_summary['AR(2)'] = evaluate(test, ar_pred, 'AR(2)')

# Plot
plt.figure(figsize=(14, 5))
plt.plot(train.index[-24:], train[-24:], label='Train', color='steelblue')
plt.plot(test.index, test, label='Actual Test', color='green')
plt.plot(test.index, ar_pred, label='AR(2) Forecast', color='red', linestyle='--')
plt.title('AR(2) Model — Forecast vs Actual')
plt.xlabel('Date'); plt.ylabel('Sales')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 8. MA Model — Moving Average Model

### MA(q) kya hota hai?
**MA(q)** predicts next value using **q past error terms** (residuals), NOT past values of y.

**Formula:** `y(t) = c + ε(t) + θ₁·ε(t-1) + θ₂·ε(t-2) + ... + θq·ε(t-q)`

Where:
- `ε(t)` = current error (white noise)
- `θ₁, θ₂, ...` = MA coefficients (learned by the model)
- `q` = MA order

### AR vs MA — Key Difference:
| | AR(p) | MA(q) |
|--|-------|-------|
| Uses | Past **values** of y | Past **errors** (residuals) |
| Memory | Long-term | Short-term shocks |
| ACF | Decays gradually | **Cuts off** at lag q |
| PACF | **Cuts off** at lag p | Decays gradually |

### Keywords:
- **ARIMA(0, 0, q)**: MA model written in ARIMA notation (p=0, d=0)
- **θ (theta)**: MA coefficients — how much past errors influence current value
- **White noise**: Pure random error with zero mean and constant variance

In [ ]:
# MA model using ARIMA(p=0, d=0, q=2)
q = 2
ma_model = ARIMA(train, order=(0, 0, q))
ma_fit = ma_model.fit()

print("MA Order:", q)
print("AIC:", round(ma_fit.aic, 2), " | BIC:", round(ma_fit.bic, 2))
print("MA Coefficients:", ma_fit.params[1:].round(4).to_dict())

# Forecast
ma_forecast = ma_fit.get_forecast(steps=len(test))
ma_pred = ma_forecast.predicted_mean
ma_pred.index = test.index

results_summary['MA(2)'] = evaluate(test, ma_pred, 'MA(2)')

# Plot
plt.figure(figsize=(14, 5))
plt.plot(train.index[-24:], train[-24:], label='Train', color='steelblue')
plt.plot(test.index, test, label='Actual', color='green')
plt.plot(test.index, ma_pred, label='MA(2) Forecast', color='purple', linestyle='--')

# Confidence interval
ci = ma_forecast.conf_int()
plt.fill_between(test.index, ci.iloc[:, 0], ci.iloc[:, 1], alpha=0.2, color='purple', label='95% CI')
plt.title('MA(2) Model — Forecast vs Actual')
plt.xlabel('Date'); plt.ylabel('Sales')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 9. ARIMA Model — AutoRegressive Integrated Moving Average

### ARIMA(p, d, q) — The Classic Forecasting Model

ARIMA **combines** AR and MA, plus handles **non-stationarity** through differencing.

**Three parameters:**
| Parameter | Meaning | How to Choose |
|-----------|---------|--------------|
| **p** | AR order — lag count for autoregression | PACF plot (where it cuts off) |
| **d** | Integration order — how many times to difference | ADF test: d=1 usually enough |
| **q** | MA order — lag count for moving average | ACF plot (where it cuts off) |

**Formula:**
```
Δᵈy(t) = c + φ₁Δᵈy(t-1)+...+φₚΔᵈy(t-p) + ε(t) + θ₁ε(t-1)+...+θqε(t-q)
```
Where `Δᵈy(t)` = d-times differenced series.

### ARIMA vs AR vs MA:
- **AR(p)** = ARIMA(p, 0, 0)
- **MA(q)** = ARIMA(0, 0, q)
- **ARIMA(p, d, q)** = combines both + differencing

### Keywords:
- **d=1**: Difference once (removes linear trend)
- **d=2**: Difference twice (removes quadratic trend, rarely needed)
- **AIC (Akaike Information Criterion)**: Model quality score — lower = better
- **BIC (Bayesian Information Criterion)**: Like AIC, penalizes complexity more

In [ ]:
# ARIMA(2, 1, 2) — from ACF/PACF + ADF results
p, d, q = 2, 1, 2
arima_model = ARIMA(train, order=(p, d, q))
arima_fit = arima_model.fit()

print(f"ARIMA({p},{d},{q}) Summary:")
print(f"  AIC: {arima_fit.aic:.2f}  |  BIC: {arima_fit.bic:.2f}")
print(f"  Log-Likelihood: {arima_fit.llf:.2f}")

# Forecast
arima_forecast = arima_fit.get_forecast(steps=len(test))
arima_pred = arima_forecast.predicted_mean
arima_pred.index = test.index

results_summary['ARIMA(2,1,2)'] = evaluate(test, arima_pred, 'ARIMA(2,1,2)')

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-24:], train[-24:], label='Train', color='steelblue')
ax.plot(test.index, test, label='Actual', color='green', linewidth=2)
ax.plot(test.index, arima_pred, label='ARIMA Forecast', color='red', linestyle='--', linewidth=2)
ci = arima_forecast.conf_int()
ax.fill_between(test.index, ci.iloc[:,0], ci.iloc[:,1], alpha=0.15, color='red', label='95% CI')
ax.set_title('ARIMA(2,1,2) — Forecast vs Actual')
ax.set_xlabel('Date'); ax.set_ylabel('Sales')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Residual diagnostics
residuals = arima_fit.resid
print(f"\nResidual Mean: {residuals.mean():.4f} (should be ~0)")
print(f"Residual Std:  {residuals.std():.4f}")

---
## 10. SARIMA Model — Seasonal ARIMA

### SARIMA(p,d,q)(P,D,Q,m) — For Data with Seasonality

SARIMA extends ARIMA by adding **seasonal components**.

**Parameters:**
| Parameter | Meaning |
|-----------|---------|
| p, d, q | Non-seasonal AR, differencing, MA (same as ARIMA) |
| **P** | Seasonal AR order |
| **D** | Seasonal differencing order |
| **Q** | Seasonal MA order |
| **m** | Seasonal period (12=monthly, 4=quarterly, 7=weekly) |

**How seasonal differencing works:**
- Non-seasonal diff: `Δy(t) = y(t) - y(t-1)`
- **Seasonal diff (D=1):** `ΔₘY(t) = y(t) - y(t-m)` → removes seasonal pattern

### When to use SARIMA:
- ACF shows spikes at seasonal lags (12, 24, 36 for monthly data)
- PACF shows spikes at seasonal lags
- After regular differencing (d=1), data still has seasonal patterns

### Keywords:
- **SARIMAX**: statsmodels class (X = supports exogenous variables, we set exog=None)
- **trend='c'**: Include constant (intercept)
- **seasonal_order=(P,D,Q,m)**: The seasonal part of the model

In [ ]:
# Check for seasonality in ACF at seasonal lags
series_diff = df['sales_diff1'].dropna()
acf_vals = acf(series_diff, nlags=36)
print("ACF at seasonal lags (12, 24, 36):")
print(f"  Lag 12: {acf_vals[12]:.3f}")
print(f"  Lag 24: {acf_vals[24]:.3f}")
print(f"  Lag 36: {acf_vals[36]:.3f}")
print("→ Significant values at lag 12 suggest seasonality (P=1 or Q=1)")

# SARIMA(1,1,1)(1,1,1,12) — common starting point for monthly data
sarima_model = SARIMAX(train,
                       order=(1, 1, 1),
                       seasonal_order=(1, 1, 1, 12),
                       trend='c')
sarima_fit = sarima_model.fit(disp=False)

print(f"\nSARIMA(1,1,1)(1,1,1,12):")
print(f"  AIC: {sarima_fit.aic:.2f}  |  BIC: {sarima_fit.bic:.2f}")

# Forecast
sarima_forecast = sarima_fit.get_forecast(steps=len(test))
sarima_pred = sarima_forecast.predicted_mean
sarima_pred.index = test.index

results_summary['SARIMA(1,1,1)(1,1,1,12)'] = evaluate(test, sarima_pred, 'SARIMA(1,1,1)(1,1,1,12)')

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-36:], train[-36:], label='Train', color='steelblue')
ax.plot(test.index, test, label='Actual', color='green', linewidth=2)
ax.plot(test.index, sarima_pred, label='SARIMA Forecast', color='darkorange', linestyle='--', linewidth=2)
ci = sarima_forecast.conf_int()
ax.fill_between(test.index, ci.iloc[:,0], ci.iloc[:,1], alpha=0.15, color='orange', label='95% CI')
ax.set_title('SARIMA(1,1,1)(1,1,1,12) — Forecast vs Actual')
ax.set_xlabel('Date'); ax.set_ylabel('Sales')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 11. Model Comparison — All Metrics Together

### Evaluation Metrics Recap:
| Metric | Formula | Best Value | Use When |
|--------|---------|------------|---------|
| **MAE** | mean(|y - ŷ|) | Lower = better | Outliers exist, easy to interpret |
| **RMSE** | √mean((y-ŷ)²) | Lower = better | Large errors are costly |
| **MAPE** | mean(|y-ŷ|/|y|)×100 | Lower = better | Comparing across different scales |

In [ ]:
# Summary comparison table
metrics_df = pd.DataFrame(results_summary).T.round(3)
print("=" * 50)
print("MODEL COMPARISON — All Metrics")
print("=" * 50)
print(metrics_df)
print("=" * 50)
print("\nBest MAE model:", metrics_df['MAE'].idxmin())
print("Best RMSE model:", metrics_df['RMSE'].idxmin())
print("Best MAPE model:", metrics_df['MAPE'].idxmin())

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    metrics_df[metric].plot(kind='bar', ax=ax, color=['#2196F3','#9C27B0','#F44336','#FF9800'],
                             edgecolor='black')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.set_xticklabels(metrics_df.index, rotation=20, ha='right')
    ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Model Comparison — Lower is Better', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 12. Future Forecasting — Beyond the Test Set

### Forecasting future (unseen) values:

After comparing models, use the **best model** trained on **ALL available data** to forecast future.

### Keywords:
- **refit on full data**: Retrain model on entire dataset (train + test) for final forecasts
- **steps**: How many future time points to forecast
- `get_forecast(steps=n)`: Returns forecast + confidence intervals

In [ ]:
# Best model: SARIMA — retrain on FULL dataset
best_model = SARIMAX(df['sales'],
                     order=(1, 1, 1),
                     seasonal_order=(1, 1, 1, 12),
                     trend='c')
best_fit = best_model.fit(disp=False)

# Forecast next 12 months
future_steps = 12
future_forecast = best_fit.get_forecast(steps=future_steps)
future_pred = future_forecast.predicted_mean
future_dates = pd.date_range(start=df.index[-1] + pd.DateOffset(months=1),
                              periods=future_steps, freq='MS')
future_pred.index = future_dates

# Confidence intervals
future_ci = future_forecast.conf_int()
future_ci.index = future_dates

print("Future Forecast (next 12 months):")
print(pd.DataFrame({'Forecast': future_pred.round(2),
                    'Lower 95%': future_ci.iloc[:,0].round(2),
                    'Upper 95%': future_ci.iloc[:,1].round(2)}))

# Plot
plt.figure(figsize=(14, 5))
plt.plot(df.index[-36:], df['sales'][-36:], label='Historical', color='steelblue')
plt.plot(future_pred.index, future_pred, label='Future Forecast', color='red', linestyle='--', linewidth=2)
plt.fill_between(future_pred.index, future_ci.iloc[:,0], future_ci.iloc[:,1],
                  alpha=0.2, color='red', label='95% CI')
plt.axvline(df.index[-1], linestyle='--', color='gray', alpha=0.7, label='Forecast Start')
plt.title('SARIMA — 12-Month Future Forecast')
plt.xlabel('Date'); plt.ylabel('Sales')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 13. Capstone Project — Monthly Product Revenue Forecasting

### Scenario:
An e-commerce company wants to forecast **monthly revenue** for the next 6 months
to plan inventory and staffing. We have 8 years of monthly revenue data.

### Workflow:
1. Generate synthetic monthly revenue data
2. EDA + decomposition
3. Stationarity check
4. Fit ARIMA & SARIMA
5. Compare → choose best → forecast 6 months ahead

In [ ]:
np.random.seed(99)
n_cap = 96  # 8 years

dates_cap = pd.date_range('2016-01-01', periods=n_cap, freq='MS')
t_cap = np.arange(n_cap)

# Revenue components
trend_cap    = 200 + 1.5 * t_cap
seasonal_cap = 60 * np.sin(2 * np.pi * t_cap / 12)
holiday_cap  = np.where((t_cap % 12) == 11, 100, 0)  # Dec spike
noise_cap    = np.random.normal(0, 15, n_cap)

revenue = trend_cap + seasonal_cap + holiday_cap + noise_cap
revenue_ts = pd.Series(revenue, index=dates_cap, name='revenue')

print("Revenue Data Summary:")
print(f"  Period: {dates_cap[0].date()} to {dates_cap[-1].date()}")
print(f"  Mean: ₹{revenue_ts.mean():.0f} | Min: ₹{revenue_ts.min():.0f} | Max: ₹{revenue_ts.max():.0f}")

# Decomposition
decomp = seasonal_decompose(revenue_ts, model='additive', period=12)
decomp.plot()
plt.suptitle('Revenue Decomposition', y=1.01, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Stationarity
adf_test(revenue_ts, 'Revenue (original)')
adf_test(revenue_ts.diff().dropna(), 'Revenue (1st diff)')

# Train-test split
split_cap = int(len(revenue_ts) * 0.8)
train_cap = revenue_ts[:split_cap]
test_cap  = revenue_ts[split_cap:]

# Fit SARIMA (best for seasonal data)
sarima_cap = SARIMAX(train_cap, order=(1,1,1), seasonal_order=(1,1,1,12), trend='c')
sarima_cap_fit = sarima_cap.fit(disp=False)

# Evaluate
cap_pred = sarima_cap_fit.get_forecast(steps=len(test_cap)).predicted_mean
cap_pred.index = test_cap.index
evaluate(test_cap, cap_pred, 'SARIMA Capstone')

In [ ]:
# Forecast future 6 months on full data
full_cap = SARIMAX(revenue_ts, order=(1,1,1), seasonal_order=(1,1,1,12), trend='c')
full_fit = full_cap.fit(disp=False)

future = full_fit.get_forecast(steps=6)
future_pred = future.predicted_mean
future_ci_cap = future.conf_int()
future_pred.index = pd.date_range(revenue_ts.index[-1]+pd.DateOffset(months=1),
                                   periods=6, freq='MS')
future_ci_cap.index = future_pred.index

print("\n6-Month Revenue Forecast (₹):")
print(pd.DataFrame({
    'Month': future_pred.index.strftime('%b %Y'),
    'Forecast': future_pred.round(0).astype(int),
    'Lower CI': future_ci_cap.iloc[:,0].round(0).astype(int),
    'Upper CI': future_ci_cap.iloc[:,1].round(0).astype(int)
}).to_string(index=False))

# Final plot
plt.figure(figsize=(14, 5))
plt.plot(revenue_ts.index[-24:], revenue_ts[-24:], label='Historical', color='steelblue')
plt.plot(future_pred.index, future_pred, 'r--o', label='6M Forecast', linewidth=2)
plt.fill_between(future_pred.index, future_ci_cap.iloc[:,0], future_ci_cap.iloc[:,1],
                  alpha=0.2, color='red')
plt.axvline(revenue_ts.index[-1], linestyle='--', color='gray', alpha=0.7)
plt.title('E-Commerce Revenue — 6-Month Forecast')
plt.xlabel('Date'); plt.ylabel('Revenue (₹)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()